# ELT & Ingestion — Core Data Operations

**Module 3 | Fundamentals Training**

You will load CSV and JSON files, explore them, apply basic PySpark transformations and write the result to a Delta table. Every PySpark step is shown next to its one-line SQL equivalent. The module starts with the **Medallion Architecture**, the Bronze/Silver/Gold pattern used in every Lakehouse project.


## Topics

| # | Topic | PySpark | SQL equivalent |
|---|-------|---------|----------------|
| 0 | Python you need for PySpark | variables, method chaining, `F.col()` | — |
| 1 | Loading CSV files | `spark.read.format("csv")` | `read_files(..., format => 'csv')` |
| 2 | Loading JSON files | `spark.read.format("json")` | `read_files(..., format => 'json')` |
| 3 | Data exploration | `display()`, `show()`, `printSchema()`, `describe()`, `summary()` | `SELECT ... LIMIT`, `DESCRIBE`, `count` / `avg` / `min` / `max` |
| 4 | Reading files in SQL | — | `read_files()` |
| 5 | Writing to Delta | `df.write.mode("overwrite").saveAsTable()` | `CREATE OR REPLACE TABLE ... AS SELECT` |
| 6 | Basic transformations | `select()`, `withColumn()`, `cast()`, `when()`, `filter()` | `SELECT`, `CAST`, `CASE WHEN`, `WHERE` |
| 7 | Aggregations, temp views and SQL | `groupBy().agg()`, `createOrReplaceTempView()`, `spark.sql()` | `GROUP BY`, `CREATE TEMP VIEW` |
| ⊕ | BONUS | Parquet, Excel, reader options, more column operations, joins, global temp views | |

> **Data used:** `customers.csv` (10 000 rows), `orders_batch.json` (100 000 rows); BONUS: `products.parquet`, `customers_extented.xlsx`

## Learning Objectives

After completing this module you will be able to:

- **Read** basic PySpark code: variables, method chaining, `F.col()`
- **Load** CSV and JSON files with `spark.read`, with an inferred schema and with an explicit schema (DDL string or `StructType`)
- **Explore** data using `display()`, `show()`, `printSchema()`, `describe()`, and `summary()`
- **Read files in SQL** with `read_files()`
- **Write** a DataFrame to a Delta table with `saveAsTable()` and name its SQL equivalent (CTAS)
- **Select and filter** columns and rows
- **Add columns** with `withColumn()`, **convert types** with `cast()` / `try_cast()`, and **apply conditional logic** with `when()` / `otherwise()`
- **Aggregate** data with `groupBy().agg()`
- **Register temporary views** and query them with SQL

> **Note:** everything after the **Summary** is ⊕ BONUS material for self-study.

## Setup

Running the setup notebook and configuring paths for all data sources used in this module.

---

In [0]:
%run ../setup/00_setup

### Configuration

Defining file paths for customers, orders, and products datasets.

In [0]:
# Paths to data directories (subdirectories in DATASET_PATH from 00_setup)
CUSTOMERS_PATH = f"{DATASET_PATH}/customers"
ORDERS_PATH = f"{DATASET_PATH}/orders"
PRODUCTS_PATH = f"{DATASET_PATH}/products"

# Paths to specific files
CUSTOMERS_CSV = f"{CUSTOMERS_PATH}/customers.csv"
ORDERS_JSON = f"{ORDERS_PATH}/orders_batch.json"
PRODUCTS_PARQUET = f"{PRODUCTS_PATH}/products.parquet"
EXCEL_PATH = f"{DATASET_PATH}/customers/customers_extented.xlsx"  # Note: filename has typo 'extented'

### Imports

`F` gives access to all Spark SQL functions (`F.col`, `F.when`, `F.sum`, ...). The types are needed only for the `StructType` schema demo.

In [0]:
# All column functions in one namespace: F.col, F.upper, F.when, F.sum, F.avg ...
# F.sum is Spark's SUM; it does not replace Python's built-in sum()
from pyspark.sql import functions as F

# Types for the StructType schema demo
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType

---
## Python You Need for PySpark (5 min)

You only need to *read* a few Python patterns. Each one has a SQL counterpart.

| Python | What it means | SQL analogy |
|---|---|---|
| `min_amount = 100` | A variable: a name for a value | `DECLARE VARIABLE min_amount = 100` |
| `orders = spark.read...` | A variable can hold a DataFrame: a query over files or a table, not a copy of the data | a view |
| `orders.filter(...)` | A method call: `object.method(arguments)` | a clause applied to a table |
| `(orders .filter(...) .select(...))` | Method chaining: one step per line, the whole chain wrapped in parentheses | `SELECT ... FROM ... WHERE ...` |
| `"total_amount"` vs `F.col("total_amount")` | A string is only a column name; `F.col()` is a column expression you can compute with, e.g. `F.col("total_amount") * 2` | `total_amount * 2` |
| `df = df.withColumn(...)` | A DataFrame never changes in place. Each method returns a **new** DataFrame, so assign the result to keep it | a chain of CTEs: `WITH step2 AS (SELECT ... FROM step1)` |
| `None`, `True`, `False` | Python names for `NULL`, `TRUE`, `FALSE` | |
| `# comment` | A comment | `-- comment` |

**Conditions:** `==` compares (a single `=` assigns a variable). `&` means AND, `|` means OR, `~` means NOT, and each condition goes in its own parentheses:
`(F.col("state") == "NY") & (F.col("total_amount") > 100)` is `state = 'NY' AND total_amount > 100`.

In [ ]:
min_amount = 100                                            # a Python variable

orders_raw = spark.read.format("json").load(ORDERS_JSON)    # a DataFrame variable

big_orders = (                                              # parentheses: one step per line
    orders_raw
    .filter(F.col("total_amount") > min_amount)             # WHERE total_amount > 100
    .select("order_id", "payment_method", "total_amount")   # SELECT order_id, payment_method, total_amount
)

big_orders.withColumn("amount_x2", F.col("total_amount") * 2)               # result discarded: big_orders is unchanged
big_orders = big_orders.withColumn("amount_x2", F.col("total_amount") * 2)  # assign the result to keep it

display(big_orders.limit(5))                                # display() is an action: now Spark runs the query

---
## Medallion Architecture — Bronze / Silver / Gold

Before we start loading data, it is worth understanding **why** we organise data into three layers. This is the Lakehouse design pattern you will see in every Databricks project.


![alt text](<../../assets/images/a433428da3144cfea92e0681ab5c73a9.png>)



### Three layers:

| Layer      | Purpose                | Transformations                  | Example                                      |
|------------|------------------------|----------------------------------|----------------------------------------------|
| **Bronze** | Raw source copy        | None or minimal                  | `customers_raw` — every row from CSV         |
| **Silver** | Cleaning & normalisation | Deduplication, types, null filtering | `customers_clean` — correct types, no duplicates |
| **Gold**   | Business / BI aggregations | GROUP BY, JOINs, metrics         | `customers_by_region` — customer count per region |

> **Key fact:** Delta Lake as the default format in Databricks means every layer has built-in change history (Time Travel), transactionality and performance.

In this module we build the **Bronze** layer: files → Delta table plus load metadata. The transformations later in the module preview what Silver does; the full Silver and Gold steps come in the final project.

![Medallion architecture](../../assets/images/555ecf8e271a4b4381567ef3e1b5a415.png)

## CSV Import (Customers)

> **Definition:** `spark.read.format("csv")` reads delimited text files. `header` = the first row holds column names; `inferSchema` = Spark scans the data to guess the column types (an extra pass over the file).
>
> **SQL:** `SELECT * FROM read_files('<path>/customers.csv', format => 'csv', header => true)`

---

### Auto Schema Inference

Using `inferSchema=true` to let Spark detect column types automatically.

In [0]:
# Load CSV data with automatic schema inference
customers_auto_df = (
    spark.read
    .format("csv")
    .option("header", "true")       # First line contains column names
    .option("inferSchema", "true")  # Automatic data type inference
    .load(CUSTOMERS_CSV)
)

In [0]:
# Verify loaded data
customers_auto_df.printSchema()   # SQL: DESCRIBE TABLE
# display() — Databricks-specific function: renders an interactive table
# Always pair with .limit(n) to avoid a full table scan
display(customers_auto_df.limit(5))

### Manual Schema Definition

An explicit schema gives predictable types and skips the inference pass. You can write it as a **DDL string** (the column list you know from `CREATE TABLE`) or as a `StructType` object. Both describe the same schema.

> **Trainer note — intentional error.** The schema in the next cell is **wrong on purpose**:
> `city` is declared before `email`, and `phone`, `state` and `customer_segment` are missing.
> A CSV reader maps columns **by position**, so the output shows e-mails in `city`, phone numbers in `email`
> and `NULL` in `registration_date` — no error is raised. Ask the group what went wrong before showing the fix below.


In [0]:
# INTENTIONALLY WRONG (live demo) — wrong column order, phone/state/customer_segment missing,
# registration_date as TimestampType. Run it, look at the output, then fix it in the next section.
customers_schema = StructType([
    StructField("customer_id", StringType(), nullable=False),
    StructField("first_name", StringType(), nullable=True),
    StructField("last_name", StringType(), nullable=True),
    StructField("city", StringType(), nullable=True),
    StructField("email", StringType(), nullable=True),
    StructField("country", StringType(), nullable=True),
    StructField("registration_date", TimestampType(), nullable=True)
])

In [0]:
# Load CSV data with manually defined schema
customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customers_schema)
    .load(CUSTOMERS_CSV)
)

In [0]:
# Verify loaded data
customers_df.printSchema()
display(customers_df.limit(5))

#### Corrected schema

The schema below lists **all 10 columns in the CSV order** and uses `DATE` for `registration_date`.
The DDL string is the short form used in the workshop; the `StructType` below it is the same schema written as Python objects.
`customers_df` from this cell is used in the rest of the module.

> **SQL:** `read_files('<path>/customers.csv', format => 'csv', header => true, schema => 'customer_id STRING, first_name STRING, ...')`

In [0]:
# CORRECTED schema — all 10 columns in the CSV header order.
# CSV columns are matched by POSITION, not by name.

# DDL string: the same column list you would write in CREATE TABLE
customer_schema_ddl = (
    "customer_id STRING, first_name STRING, last_name STRING, email STRING, phone STRING, "
    "city STRING, state STRING, country STRING, registration_date DATE, customer_segment STRING"
)

# The same schema as a StructType (longer; you will meet it in existing code)
customer_schema = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("first_name", StringType(), nullable=True),
    StructField("last_name", StringType(), nullable=True),
    StructField("email", StringType(), nullable=True),
    StructField("phone", StringType(), nullable=True),
    StructField("city", StringType(), nullable=True),
    StructField("state", StringType(), nullable=True),
    StructField("country", StringType(), nullable=True),
    StructField("registration_date", DateType(), nullable=True),
    StructField("customer_segment", StringType(), nullable=True)
])

customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customer_schema_ddl)      # .schema(customer_schema) gives the same result
    .load(CUSTOMERS_CSV)
)
customers_df.printSchema()

## JSON Import (Orders)

> **Definition:** `spark.read.format("json")` reads JSON Lines files (one JSON object per line). The JSON reader always infers the schema, so there is no `inferSchema` option; an explicit schema skips that scan and fixes the types.
>
> **SQL:** `SELECT * FROM read_files('<path>/orders_batch.json', format => 'json')`

---

### Auto Schema Inference

Letting Spark infer the JSON schema automatically.

In [0]:
# Load JSON data with automatic schema inference
# (the JSON reader always infers the schema — no inferSchema option is needed)
orders_auto_df = (
    spark.read
    .format("json")
    .load(ORDERS_JSON)
)

In [0]:
# Verify loaded data
orders_auto_df.printSchema()
display(orders_auto_df.limit(5))

### Explicit Schema

Providing an explicit schema to select and type only the columns we need. JSON fields are matched **by name**, so the column order does not matter and fields you leave out are skipped.

In [0]:
# DDL string with the 6 columns we need; the other JSON fields are skipped
# order_datetime stays a STRING here; we convert it in the cast section
orders_schema = (
    "order_id STRING, customer_id STRING, order_datetime STRING, "
    "quantity INT, total_amount DOUBLE, payment_method STRING"
)

In [0]:
# Load JSON data with manually defined schema
orders_df = (
    spark.read
    .format("json")
    .schema(orders_schema)
    .load(ORDERS_JSON)
)

In [0]:
# Verify loaded data
orders_df.printSchema()
display(orders_df.limit(5))

## Data Exploration — `show()`, `describe()`, `summary()`

| Method | What it returns | SQL equivalent |
|---|---|---|
| `display(df)` | Interactive table with charts and Data Profile (Databricks only) | `SELECT * FROM t` in a `%sql` cell |
| `df.show(n, truncate)` | Plain-text preview of `n` rows; works in jobs and logs too | `SELECT * FROM t LIMIT n` |
| `df.printSchema()` | Column names and types | `DESCRIBE TABLE t` |
| `df.describe(*cols)` | `count`, `mean`, `stddev`, `min`, `max` | `SELECT count(c), avg(c), stddev(c), min(c), max(c) FROM t` |
| `df.summary(*stats)` | `describe()` plus percentiles `25%`, `50%`, `75%`; you can pick the statistics | `percentile(c, 0.5)` |
| `df.count()`, `df.select(c).distinct().count()` | Number of rows / distinct values | `SELECT count(*)`, `count(DISTINCT c)` |

In [ ]:
# show() — plain-text preview; truncate=False prints full values
# SQL: SELECT * FROM customers LIMIT 5
customers_df.show(5, truncate=False)

In [ ]:
# Row count and number of distinct values
# SQL: SELECT count(*), count(DISTINCT country), count(DISTINCT state) FROM customers
print(f"Rows               : {customers_df.count():,}")
print(f"Distinct countries : {customers_df.select('country').distinct().count()}")
print(f"Distinct states    : {customers_df.select('state').distinct().count()}")

In [ ]:
# describe() and summary() are meant for NUMERIC columns: use the order quantities and amounts
# SQL: SELECT count(total_amount), avg(total_amount), stddev(total_amount), min(total_amount), max(total_amount) FROM orders
display(orders_df.describe("quantity", "total_amount"))

# summary() with chosen statistics: minimum, quartiles, maximum
display(orders_df.select("quantity", "total_amount").summary("min", "25%", "50%", "75%", "max"))

# Data-quality hook: the minimum total_amount is negative (returns or broken rows).
# Bronze keeps them as they are; Silver decides what to do with them.

## read_files() — Reading Files in SQL

> **Definition:** `read_files(path, format => ...)` is a SQL table-valued function (Databricks Runtime 13.3 LTS and above) that reads files from a Unity Catalog Volume directly in a query. Every row also carries the hidden `_metadata` column (`file_path`, `file_name`, `file_size`, ...).

| | `read_files()` | `spark.read` |
|---|---|---|
| Language | SQL | Python |
| Schema | Inferred, or `schema => '...'` | Inferred, or `.schema(...)` |
| Typical use | SQL-first work, CTAS into Bronze | Python pipelines |

```sql
-- Read CSV from a Volume
SELECT * FROM read_files('/Volumes/catalog/schema/volume/file.csv', format => 'csv', header => true);
```

> **Tip:** `read_files()` is the recommended way to read files in SQL workflows with Unity Catalog.

In [ ]:
# DATASET_PATH comes from 00_setup. Declare it as a SQL session variable
# so the %sql cells below can use it without editing any path by hand.
spark.sql(f"DECLARE OR REPLACE VARIABLE dataset_path STRING DEFAULT '{DATASET_PATH}'")
print(CUSTOMERS_CSV)

In [ ]:
%sql
-- Read CSV from a Unity Catalog Volume using read_files() in SQL.
-- dataset_path is the session variable declared in the cell above.
SELECT *, _metadata.file_path FROM read_files(
  dataset_path || '/customers/customers.csv',
  format => 'csv',
  header => true
)

In [ ]:
%sql
-- Create a Bronze table straight from the files: raw columns + load metadata
CREATE OR REPLACE TABLE bronze.customers AS
SELECT *,
       current_timestamp() AS _load_ts,
       _metadata.file_path AS _source_file
FROM read_files(dataset_path || '/customers/customers.csv', format => 'csv', header => true)

In [ ]:
%sql 
SELECT * FROM bronze.customers

## Write to Delta — PySpark

The same Bronze table, built from a DataFrame. You use this pattern in Workshop Task 11 and in the final project.

| Step | PySpark | SQL equivalent (cell above) |
|---|---|---|
| Load timestamp | `.withColumn("_load_ts", F.current_timestamp())` | `current_timestamp() AS _load_ts` |
| Source file | `.withColumn("_source_file", F.col("_metadata.file_path"))` | `_metadata.file_path AS _source_file` |
| Save as a managed Delta table | `.write.mode("overwrite").saveAsTable("catalog.schema.table")` | `CREATE OR REPLACE TABLE ... AS SELECT` |

`_metadata` is a hidden column of every DataFrame read from files. Delta is the default table format, so `saveAsTable` needs no `.format("delta")`.
After the run, open **Catalog** → your catalog → `bronze` → `customers` to see the table, its schema and its history.

In [ ]:
BRONZE_CUSTOMERS = f"{CATALOG}.{BRONZE_SCHEMA}.customers"

customers_bronze = (
    customers_df                                                # raw data: no business logic, no filtering
    .withColumn("_load_ts", F.current_timestamp())              # when the row was loaded
    .withColumn("_source_file", F.col("_metadata.file_path"))   # which file it came from
)

(
    customers_bronze.write
    .mode("overwrite")                     # replace the table content on every run
    .option("overwriteSchema", "true")     # needed when the column list or types change between runs
    .saveAsTable(BRONZE_CUSTOMERS)
)

display(spark.table(BRONZE_CUSTOMERS).limit(5))

## DataFrame Transformations

Each transformation returns a **new** DataFrame; the original does not change. Every example comes with its SQL equivalent.
The SQL lines use the views `customers` and `orders`, which we register in *Temporary Views & SQL* below.

---

### select — Choosing Columns

> **Definition:** `df.select(...)` keeps only the listed columns or expressions.
>
> **SQL:** `SELECT customer_id, upper(first_name) AS first_name_upper, email FROM customers`

In [0]:
# Select specific columns
customers_selected = customers_df.select("customer_id", "first_name", "last_name", "email")
display(customers_selected.limit(5))

# Select with column expressions
customers_transformed = customers_df.select(
    F.col("customer_id"),
    F.upper(F.col("first_name")).alias("first_name_upper"),   # .alias() = AS
    F.col("email")
)
display(customers_transformed.limit(5))

### withColumn — Adding/Modifying Columns

> **Definition:** `df.withColumn("name", expression)` adds a column, or replaces it when the name already exists.
>
> **SQL:** `SELECT *, concat_ws(' ', first_name, last_name) AS full_name, year(registration_date) AS registration_year FROM customers`

In [0]:
# Add a new column. concat_ws(separator, ...) joins strings and skips NULL values
customers_with_fullname = customers_df.withColumn(
    "full_name",
    F.concat_ws(" ", F.col("first_name"), F.col("last_name"))
)
display(customers_with_fullname.select("customer_id", "first_name", "last_name", "full_name").limit(5))

# Add several columns: chain withColumn calls inside parentheses
customers_enriched = (
    customers_df
    .withColumn("email_lower", F.lower(F.col("email")))
    .withColumn("registration_year", F.year(F.col("registration_date")))
)
display(customers_enriched.select("customer_id", "email", "email_lower", "registration_date", "registration_year").limit(5))

### cast — Type Conversion

> **Definition:** `F.col("x").cast("date")` converts a column to another type (`"int"`, `"double"`, `"date"`, `"timestamp"`, ...).
>
> **SQL:** `CAST(registration_date AS DATE)`

**ANSI mode.** On Databricks Runtime 17.0 and above and on serverless compute, ANSI mode is on: a value that cannot be converted makes the query fail with `CAST_INVALID_INPUT` instead of becoming `NULL`. When bad values are expected, use `try_cast`, which returns `NULL` for them.

| | PySpark | SQL |
|---|---|---|
| Strict: fails on a bad value | `F.col("x").cast("double")` | `CAST(x AS DOUBLE)` |
| Tolerant: `NULL` for a bad value | `F.col("x").try_cast("double")` | `try_cast(x AS DOUBLE)` |

In [0]:
# Without a schema every CSV column is a STRING
customers_raw_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(CUSTOMERS_CSV)
)

# STRING -> DATE. SQL: CAST(registration_date AS DATE)
customers_typed = customers_raw_df.withColumn("registration_date", F.col("registration_date").cast("date"))
customers_raw_df.select("registration_date").printSchema()   # string
customers_typed.select("registration_date").printSchema()    # date

# STRING -> TIMESTAMP. order_datetime looks like 2024-12-31T23:56:00. SQL: CAST(order_datetime AS TIMESTAMP)
# For other formats use F.to_timestamp(F.col("x"), "dd/MM/yyyy HH:mm")
orders_ts = orders_df.withColumn("order_ts", F.col("order_datetime").cast("timestamp"))
orders_ts.select("order_datetime", "order_ts").printSchema()
display(orders_ts.select("order_id", "order_datetime", "order_ts").limit(5))

In [ ]:
# phone holds text such as "+48 694 026 542", which is not a number.
# With ANSI mode on (DBR 17.0+, serverless) CAST fails with CAST_INVALID_INPUT.
try:
    customers_raw_df.select(F.col("phone").cast("double")).limit(5).collect()
    print("No error: ANSI mode is off on this compute, so the bad values became NULL")
except Exception as e:
    print("Query failed:", (str(e).splitlines() or [str(e)])[0][:200])

In [ ]:
# try_cast returns NULL instead of failing. SQL: SELECT phone, try_cast(phone AS DOUBLE) FROM customers
phone_check = customers_raw_df.select(
    "phone",
    F.col("phone").try_cast("double").alias("phone_as_double")
)
display(phone_check.limit(5))

### when / otherwise — Conditional Column Logic

> **Definition:** `F.when(condition, value)` is SQL `CASE WHEN`. Chain more `.when(...)` branches; `.otherwise(default)` is the `ELSE` (without it, unmatched rows get `NULL`).

| Pattern | SQL equivalent |
|---|---|
| `F.when(cond, val)` | `CASE WHEN cond THEN val END` |
| `F.when(c1, v1).when(c2, v2)` | `CASE WHEN c1 THEN v1 WHEN c2 THEN v2 END` |
| `F.when(c1, v1).otherwise(v2)` | `CASE WHEN c1 THEN v1 ELSE v2 END` |
| `F.when(F.col("x").isNull(), F.lit("N/A")).otherwise(F.col("x"))` | `COALESCE(x, 'N/A')` |

In [0]:
# Classify customers by registration year
# SQL: SELECT *, CASE WHEN year(registration_date) >= 2025 THEN 'New'
#                     WHEN year(registration_date) >= 2024 THEN 'Regular'
#                     ELSE 'Loyal' END AS customer_tier
#      FROM customers
customers_classified = customers_df.withColumn(
    "customer_tier",
    F.when(F.year(F.col("registration_date")) >= 2025, "New")
    .when(F.year(F.col("registration_date")) >= 2024, "Regular")
    .otherwise("Loyal")
)
display(customers_classified.select("customer_id", "first_name", "registration_date", "customer_tier").limit(10))
display(customers_classified.groupBy("customer_tier").count().orderBy("customer_tier"))

# Replace NULL with a default value. SQL: SELECT customer_id, city, coalesce(city, 'Unknown') AS city_safe FROM customers
customers_safe = customers_df.withColumn(
    "city_safe",
    F.when(F.col("city").isNull(), F.lit("Unknown")).otherwise(F.col("city"))
)
# show rows where city was NULL, so the replacement is visible
display(customers_safe.filter(F.col("city").isNull()).select("customer_id", "city", "city_safe").limit(5))

## Filtering Data

### Simple Conditions

> **Definition:** `df.filter(condition)` keeps the rows that meet the condition. `where()` is the same method under its SQL name.
>
> **SQL:** `SELECT * FROM customers WHERE state = 'NY'`

In [0]:
# Filter by state (every customer is from the USA, so a country filter would keep all rows)
ny_customers = customers_df.filter(F.col("state") == "NY")
display(ny_customers.select("customer_id", "first_name", "last_name", "state").limit(5))

# where() is the same as filter(); a SQL condition string also works
nyc_customers = customers_df.where("city = 'New York'")
display(nyc_customers.select("customer_id", "first_name", "city", "state").limit(5))

### Multiple Conditions (`&` = AND, `|` = OR)

> **Definition:** combine conditions with `&` (AND), `|` (OR) and `~` (NOT), and wrap **each** condition in parentheses; without them Python evaluates the operators in the wrong order.
>
> **SQL:** `SELECT * FROM customers WHERE state = 'NY' AND year(registration_date) = 2025`

In [0]:
# AND condition
ny_2025 = customers_df.filter(
    (F.col("state") == "NY") & (F.year(F.col("registration_date")) == 2025)
)
display(ny_2025.select("customer_id", "first_name", "state", "registration_date").limit(5))

# OR condition. SQL: WHERE state = 'NY' OR state = 'CA'
ny_or_ca = customers_df.filter(
    (F.col("state") == "NY") | (F.col("state") == "CA")
)
display(ny_or_ca.groupBy("state").count())

### Null Handling (`isNull`, `isNotNull`)

> **Definition:** `F.col("x").isNull()` and `.isNotNull()` test for `NULL`. `F.col("x") == None` does not work, just as `x = NULL` does not work in SQL.
>
> **SQL:** `SELECT * FROM customers WHERE city IS NULL`

In [0]:
# Filter rows where email is NOT null. SQL: WHERE email IS NOT NULL
customers_with_email = customers_df.filter(F.col("email").isNotNull())

# Filter rows where city IS null. SQL: WHERE city IS NULL
customers_no_city = customers_df.filter(F.col("city").isNull())
if customers_no_city.count() > 0:
    display(customers_no_city.select("customer_id", "first_name", "city", "country").limit(5))

## Aggregations

> **Definition:** `df.groupBy("col").agg(...)` is `GROUP BY`: one row per group, one column per aggregate function. Name each result with `.alias(...)`.
>
> **SQL:** `SELECT payment_method, SUM(total_amount) AS total_revenue, AVG(total_amount) AS avg_order_value, COUNT(*) AS order_count FROM orders GROUP BY payment_method ORDER BY total_revenue DESC`

In [0]:
# Count by state. SQL: SELECT state, COUNT(*) AS count FROM customers GROUP BY state ORDER BY count DESC
# Raw data: lower-case codes (ny, nj) form their own groups; Silver would standardise them
customers_by_state = customers_df.groupBy("state").count().orderBy(F.desc("count"))
display(customers_by_state.limit(10))

In [0]:
# Sum, average and count in one agg() call
revenue_by_payment = (
    orders_df
    .groupBy("payment_method")
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_order_value"),
        F.count("*").alias("order_count")
    )
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_payment)

## Temporary Views & SQL

> **Definition:** `df.createOrReplaceTempView("name")` gives a DataFrame a name you can use in SQL, in `spark.sql(...)` or in a `%sql` cell. It stores the query, not a copy of the data, and exists only in the current session (this notebook).
>
> **SQL:** `CREATE OR REPLACE TEMP VIEW name AS SELECT ...`

| View Type | Scope | Persistence | Syntax |
|---|---|---|---|
| **Temp View** | Current session | Session only | `CREATE TEMP VIEW` |
| **Global Temp View** (⊕ BONUS) | All sessions on a classic cluster | Until cluster restart | `CREATE GLOBAL TEMP VIEW` → `global_temp.name` |
| **Permanent View** | Unity Catalog | Persistent | `CREATE VIEW catalog.schema.name` |

> **Tip:** temp views are not visible in other notebooks. A view stores the query definition, not data.

In [0]:
# Create temporary views from our DataFrames
customers_df.createOrReplaceTempView("customers")
orders_df.createOrReplaceTempView("orders")

print("Views customers and orders are available for SQL in this session.")

### SQL Queries via spark.sql()

Executing SQL statements against registered temporary views.

In [0]:
# Orders per customer in SQL. CUST999999 and NULL at the top are placeholders in the raw data
result = spark.sql("""
    SELECT
        customer_id,
        COUNT(*) as order_count,
        SUM(total_amount) as total_spent
    FROM orders
    GROUP BY customer_id
    ORDER BY order_count DESC
    LIMIT 10
""")

display(result)

### DataFrame API vs SQL — Same Result

Demonstrating that both approaches produce identical results under the same Catalyst optimizer.

In [0]:
# Same query, two approaches

# DataFrame API
df_api_result = (
    customers_df
    .groupBy("city")
    .count()
    .orderBy(F.desc("count"))
    .limit(5)
)

display(df_api_result)

In [0]:
# SQL
sql_result = spark.sql("""
    SELECT 
        city,
        COUNT(*) as count
    FROM customers
    GROUP BY city
    ORDER BY count DESC
    LIMIT 5
""")

display(sql_result)

In [ ]:
%sql
-- The same query in a %sql cell: temp views are visible to SQL cells in this notebook
SELECT city, COUNT(*) AS count
FROM customers
GROUP BY city
ORDER BY count DESC
LIMIT 5

---

## Summary

| Topic | Key Takeaway | SQL equivalent |
|---|---|---|
| **Python** | `df = df.method(...)`: each step returns a new DataFrame; `F.col()` builds column expressions | |
| **Schema** | An explicit schema (DDL string or `StructType`) gives predictable types and skips inference | `read_files(..., schema => '...')` |
| **Formats** | CSV columns are matched by position, JSON fields by name | `read_files(..., format => ...)` |
| **Exploration** | `display`, `show`, `printSchema`; `describe` / `summary` on numeric columns | `DESCRIBE`, `count` / `avg` / `min` / `max` |
| **Bronze** | Raw data + `_load_ts` + `_source_file`, written with `saveAsTable` | `CREATE OR REPLACE TABLE ... AS SELECT` |
| **Transformations** | `select`, `withColumn`, `cast`, `when` | `SELECT`, `CAST`, `CASE WHEN` |
| **ANSI mode** | DBR 17.0+ and serverless: a bad cast fails with `CAST_INVALID_INPUT` | `try_cast` returns `NULL` |
| **Filtering** | `filter` / `where`, each condition in parentheses, `isNull` | `WHERE`, `IS NULL` |
| **Aggregations** | `groupBy().agg()` with `.alias()` | `GROUP BY` |
| **Views** | A temp view names a DataFrame for SQL in this session | `CREATE TEMP VIEW` |

---


← [Spark Architecture](02_spark_architecture.ipynb) | [Delta Lake →](04_delta_fundamentals.ipynb)

---
> ⊕ **BONUS — Extended CSV Reader Options**
>
> Skip in standard schedule. This table lists the CSV options (sep, escape, nullValue, etc.).
> It is reference material; point participants to the Databricks documentation instead.
---

### Extended Reader Options

All important CSV options for production use — go beyond the basics:

| Option | Type | Default | Description |
|---|---|---|---|
| `header` | bool | `false` | First row as column names |
| `inferSchema` | bool | `false` | Auto-detect types (2 scans — expensive!) |
| `delimiter` / `sep` | string | `,` | Column separator (e.g. `;`, `\t`) |
| `quote` | string | `"` | Quote character for text fields |
| `escape` | string | `\` | Escape character inside quoted strings |
| `nullValue` | string | `""` | String to treat as `null` (e.g. `"N/A"`) |
| `emptyValue` | string | `""` | Value to replace empty strings |
| `nanValue` | string | `NaN` | String to treat as `NaN` |
| `dateFormat` | string | `yyyy-MM-dd` | Pattern for `DateType` columns |
| `timestampFormat` | string | ISO 8601 | Pattern for `TimestampType` columns |
| `encoding` | string | `UTF-8` | File encoding (e.g. `ISO-8859-1`, `UTF-16`) |
| `multiLine` | bool | `false` | Allow newlines inside quoted fields |
| `comment` | string | disabled | Lines starting with this char are skipped |
| `ignoreLeadingWhiteSpace` | bool | `false` | Strip leading whitespace |
| `ignoreTrailingWhiteSpace` | bool | `false` | Strip trailing whitespace |
| `mode` | string | `PERMISSIVE` | Error handling: `PERMISSIVE` / `DROPMALFORMED` / `FAILFAST` |
| `columnNameOfCorruptRecord` | string | `_corrupt_record` | Column to store malformed rows (PERMISSIVE mode) |
| `maxColumns` | int | `20480` | Max number of columns allowed |
| `samplingRatio` | double | `1.0` | Fraction of rows to scan for `inferSchema` |

```python
# Production-grade CSV read with most common options
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", ";")                          # semicolon-delimited
    .option("encoding", "UTF-8")
    .option("nullValue", "N/A")
    .option("emptyValue", "UNKNOWN")
    .option("dateFormat", "dd/MM/yyyy")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .option("mode", "DROPMALFORMED")             # silently drop bad rows
    .option("multiLine", "true")                 # allow newlines in fields
    .schema(my_schema)
    .load("/Volumes/catalog/schema/volume/data.csv")
)
```

> **Tip:** `PERMISSIVE` (default) stores bad rows in `_corrupt_record`. `FAILFAST` raises an exception on first bad row. Use `DROPMALFORMED` to silently skip them.

---
> ⊕ **BONUS — JSON Reader Options**
>
> Skip in standard schedule. Reference table for JSON options such as multiLine and mode.
> The orders file needs none of them.
---

### JSON Reader Options

| Option | Type | Default | Description |
|---|---|---|---|
| `multiLine` | bool | `false` | Allow JSON objects spanning multiple lines |
| `allowComments` | bool | `false` | Allow `//` and `/* */` comments in JSON |
| `allowUnquotedFieldNames` | bool | `false` | Allow keys without quotes |
| `allowSingleQuotes` | bool | `true` | Allow single-quoted strings |
| `allowNumericLeadingZeros` | bool | `false` | Allow `007` as integer |
| `allowNonNumericNumbers` | bool | `true` | Allow `NaN`, `INF`, `-INF` |
| `dateFormat` | string | `yyyy-MM-dd` | Pattern for date parsing |
| `timestampFormat` | string | ISO 8601 | Pattern for timestamp parsing |
| `columnNameOfCorruptRecord` | string | `_corrupt_record` | Store unparseable rows |
| `mode` | string | `PERMISSIVE` | `PERMISSIVE` / `DROPMALFORMED` / `FAILFAST` |
| `primitivesAsString` | bool | `false` | Read all primitives as strings |
| `dropFieldIfAllNull` | bool | `false` | Drop columns that are all null |
| `encoding` | string | `UTF-8` | File encoding |
| `samplingRatio` | double | `1.0` | Fraction of rows used to infer the schema |

```python
# Multi-line JSON (one object across multiple lines)
df = (
    spark.read
    .format("json")
    .option("multiLine", "true")
    .option("mode", "DROPMALFORMED")
    .schema(my_schema)
    .load("/Volumes/catalog/schema/volume/data.json")
)
```

---
> ⊕ **BONUS — Parquet Import**
>
> Skip in standard schedule. Reading Parquet is one line and no exercise uses it; Delta (M04) is where Parquet matters.
> Core message (verbal, 30 sec): *"Parquet carries its own schema — spark.read.parquet(path), no inferSchema needed."*
> Cover for: advanced groups and the 2-day format.
---

## Parquet Import (Products)

> **Definition:** `spark.read.format("parquet")` reads Parquet, the columnar file format under Delta Lake. Parquet stores the schema inside the file, so no schema or `inferSchema` is needed.
>
> **SQL:** `SELECT * FROM read_files('<path>/products.parquet', format => 'parquet')`

In [0]:
# Parquet already contains built-in schema - no need to define it
products_df = (
    spark.read
    .format("parquet")
    .load(PRODUCTS_PARQUET)
)

In [0]:
# Verify loaded data
products_df.printSchema()
display(products_df.limit(5))
products_df.createOrReplaceTempView("products")   # available for SQL as well

---
> ⊕ **BONUS — Excel Import**
>
> Skip in standard schedule. Databricks has a built-in Excel reader (Databricks Runtime 17.1 or above), but Excel is rarely
> a production source; prefer CSV or Parquet exports. Cover only if the group works with Excel files.
---

## Excel Import (Extended Customers)

> **Definition:** `spark.read.format("excel")` is the built-in reader for `.xls` and `.xlsx` files; it requires Databricks Runtime 17.1 or above and no extra library. `dataAddress` selects a sheet and cell range (e.g. `"Sheet1!A1:Z100"`); `headerRows` (0 or 1) says whether the first row of that range holds column names.

In [0]:
# Built-in Excel reader (DBR 17.1+): sheet "Arkusz1", cells D6:M26, first row = column names
customers_extended_df = (
    spark.read
    .format("excel")
    .option("dataAddress", "Arkusz1!D6:M26")
    .option("headerRows", 1)
    .load(EXCEL_PATH)
)

In [0]:
# Verify loaded data
customers_extended_df.printSchema()
display(customers_extended_df.limit(5))

---
> ⊕ **BONUS — Performance: inferSchema vs Manual Schema (Benchmark)**
>
> Skip in standard schedule. The benchmark measures wall-clock time difference between auto and manual schema.
> Core message (verbal, 30 sec): *"inferSchema scans the file twice — for large files prefer manual schema."*
> Return here for groups interested in performance tuning.
---

## Performance: inferSchema vs Manual Schema

> **Definition:** with `inferSchema=true` Spark reads the data once to detect the types and again to load it. With an explicit schema it reads the data once.

Comparing read time between schema inference and an explicit schema.

In [0]:
import time

# Automatic schema inference
start_auto = time.time()
df_auto = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CUSTOMERS_CSV)
)
count_auto = df_auto.count()  # Action - forces execution
time_auto = time.time() - start_auto

In [0]:
# Manual schema definition
start_manual = time.time()
df_manual = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customer_schema)  # the corrected 10-column schema
    .load(CUSTOMERS_CSV)
)
count_manual = df_manual.count()  # Action - forces execution
time_manual = time.time() - start_manual

In [0]:
# Comparison
speedup = (time_auto - time_manual) / time_auto * 100
print(f"\nConclusion: Manual schema is faster by {speedup:.1f}%")

---
> ⊕ **BONUS — More Column and Row Operations: withColumnRenamed, drop, distinct, orderBy, isin**
>
> Skip in standard schedule. Each one maps 1:1 to SQL the group already knows (`AS`, leaving a column out of `SELECT`,
> `DISTINCT`, `ORDER BY`, `IN`). Show them when participants ask how to write these SQL steps in PySpark.
---

### withColumnRenamed — Renaming

> **Definition:** `df.withColumnRenamed("old", "new")` renames one column. To rename several, use `select()` with `.alias()`.
>
> **SQL:** `SELECT customer_id AS id, ... FROM customers`

In [0]:
# Rename single column
customers_renamed = customers_df.withColumnRenamed("customer_id", "id")
display(customers_renamed.limit(5))

# Rename multiple columns using select with alias
customers_multi_renamed = customers_df.select(
    F.col("customer_id").alias("id"),
    F.col("first_name").alias("fname"),
    F.col("last_name").alias("lname"),
    F.col("email"),
    F.col("city")
)
display(customers_multi_renamed.limit(5))

### drop — Removing Columns

> **Definition:** `df.drop("a", "b")` returns the DataFrame without the listed columns; a name that does not exist is ignored.
>
> **SQL:** `SELECT * EXCEPT (email, city, country) FROM customers`

In [0]:
# Drop single column
customers_dropped = customers_df.drop("email")

# Drop multiple columns
customers_minimal = customers_df.drop("email", "city", "country")
display(customers_minimal.limit(5))

### distinct / dropDuplicates — Unique Rows

> **Definition:** `df.distinct()` keeps unique rows (all columns compared); `df.dropDuplicates(["a", "b"])` compares only the listed columns.
>
> **SQL:** `SELECT DISTINCT state FROM customers`

In [0]:
# Get distinct states (every customer in this dataset is from the USA, so country has one value)
distinct_states = customers_df.select("state").distinct()
display(distinct_states.orderBy("state"))

# Drop duplicates based on specific columns
# Raw data: note the mixed spellings (California / CALIFORNIA, CA / ca) — cleaning belongs to Silver
unique_locations = customers_df.select("city", "state").dropDuplicates(["city", "state"])
display(unique_locations.orderBy("state", "city").limit(10))


### orderBy — Sorting

> **Definition:** `df.orderBy(...)` (alias `sort()`) sorts rows, ascending by default; use `F.desc("col")` for descending. A global sort moves data between machines, so sort as late as possible.
>
> **SQL:** `SELECT * FROM customers ORDER BY country ASC, registration_date DESC`

In [0]:
# Sort by single column (ascending)
customers_sorted_asc = customers_df.orderBy("registration_date")
display(customers_sorted_asc.select("customer_id", "first_name", "last_name", "registration_date").limit(5))

# Sort by multiple columns with different directions
customers_sorted_multi = customers_df.orderBy(F.asc("country"), F.desc("registration_date"))
display(customers_sorted_multi.select("customer_id", "first_name", "country", "registration_date").limit(10))

### isin — Filter by List

> **Definition:** `F.col("x").isin([...])` is true when the value is in the list; negate it with `~`.
>
> **SQL:** `SELECT * FROM customers WHERE state IN ('NY', 'CA', 'TX', 'FL')`

In [0]:
# Filter by list of states
selected_states = ["NY", "CA", "TX", "FL"]
customers_selected_states = customers_df.filter(F.col("state").isin(selected_states))

# Show distribution by state
display(customers_selected_states.groupBy("state").count().orderBy(F.desc("count")))

---
> ⊕ **BONUS — fillna / dropna — Replacing and Dropping Nulls**
>
> Skip in standard schedule. `isNull` / `isNotNull` (above) cover the Fundamental use case.
> `fillna` and `dropna` are Silver-layer data quality patterns — better covered in the Silver workshop.
---

### fillna / na.drop — Handling Missing Values

> **Definition (Databricks):**
> - `DataFrame.fillna(value, subset=None)` (alias: `na.fill()`) — replaces `null` values with a given value. Can be a dict to specify different fill values per column. Works on `StringType`, numeric types, and `BooleanType`.
> - `DataFrame.dropna(how='any', thresh=None, subset=None)` (alias: `na.drop()`) — removes rows that contain null values.
>   - `how='any'` — drop if **any** specified column is null
>   - `how='all'` — drop only if **all** specified columns are null
>   - `thresh=N` — keep rows with at least N non-null values

| Method | Example | Effect |
|---|---|---|
| `fillna("Unknown")` | All nulls in string cols → `"Unknown"` | Global fill |
| `fillna(0, subset=["amount"])` | Nulls in `amount` → `0` | Column-specific |
| `fillna({"city": "Unknown", "age": 0})` | Per-column dict | Mixed types |
| `dropna()` | Drop rows with any null | Strict |
| `dropna(how="all")` | Drop rows where all are null | Lenient |
| `dropna(subset=["customer_id"])` | Only check these columns | Targeted |


---
> ⊕ **BONUS — String Operations**
>
> Skip in standard schedule. `like` / `contains` / `startswith` are simple filters the group can look up when needed.
> Core message (verbal, 30 sec): *"Column has string helpers: like, contains, startswith — same as SQL LIKE."*
> Cover for: advanced groups and the 2-day format.
---

### String Operations (`like`, `contains`, `startswith`)

> **Definition (Databricks):**
> - `Column.like(pattern)` — matches a SQL wildcard pattern: `%` (any string), `_` (one character). SQL equivalent: `LIKE`.
> - `Column.contains(other)` — returns `true` if the column contains the substring `other`.
> - `Column.startswith(other)` / `endswith(other)` — checks prefix / suffix.
> - `Column.rlike(pattern)` — matches using a regular expression (Java regex).
>
> All string operations return `null` if the column value is `null`.

Pattern matching and string-based filtering on column values.


In [0]:
# Filter using like (SQL-style pattern matching). SQL: WHERE email LIKE '%@gmail.com'
gmail_customers = customers_df.filter(F.col("email").like("%@gmail.com"))
display(gmail_customers.select("customer_id", "first_name", "email").limit(5))

# Filter using contains. SQL: WHERE city LIKE '%New%'
new_cities = customers_df.filter(F.col("city").contains("New"))
display(new_cities.select("customer_id", "first_name", "city").limit(5))

# Filter using startswith. SQL: WHERE first_name LIKE 'J%'
j_names = customers_df.filter(F.col("first_name").startswith("J"))
display(j_names.select("customer_id", "first_name", "last_name").limit(5))

---
> ⊕ **BONUS — More Aggregations: min / max, several aggregates, HAVING**
>
> Skip in standard schedule. One `groupBy().agg()` in the main part of the module covers the pattern; these are variations of it.
---

### min / max

> **Definition:** `F.min(col)` / `F.max(col)` return the smallest / largest value; `NULL` values are ignored. Without `groupBy`, `df.agg(...)` computes them over the whole DataFrame.
>
> **SQL:** `SELECT min(total_amount), max(total_amount), avg(total_amount) FROM orders`

In [0]:
# Min and max order amounts
# Negative amounts are returns or broken rows in the raw data — Silver decides how to treat them
order_stats = orders_df.agg(
    F.min("total_amount").alias("min_amount"),
    F.max("total_amount").alias("max_amount"),
    F.avg("total_amount").alias("avg_amount")
)
display(order_stats)

### Multiple Aggregations with agg()

> **Definition:** one `agg(...)` call can compute many aggregates in a single pass over the data.
>
> **SQL:** `SELECT customer_id, COUNT(*), SUM(total_amount), AVG(total_amount), MIN(total_amount), MAX(total_amount) FROM orders GROUP BY customer_id`

In [0]:
# Example: Multiple aggregations on orders by customer
# The top rows are CUST999999 (placeholder for an unknown customer) and NULL — raw-data issues, not real customers
customer_stats = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("total_amount").alias("total_spent"),
        F.avg("total_amount").alias("avg_order_value"),
        F.min("total_amount").alias("min_order"),
        F.max("total_amount").alias("max_order")
    )
    .orderBy(F.desc("total_spent"))
)

display(customer_stats.limit(10))

### HAVING Equivalent (filter after groupBy)

> **Definition:** `filter()` **before** `groupBy` is `WHERE`; `filter()` **after** `groupBy().agg()` is `HAVING`.
>
> **SQL:** `SELECT customer_id, COUNT(*) AS order_count FROM orders GROUP BY customer_id HAVING COUNT(*) > 5`

In [0]:
# Customers with more than 5 orders (CUST999999 and NULL again lead — see the note above)
high_frequency = (
    orders_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("order_count"),
        F.sum("total_amount").alias("total_spent")
    )
    .filter(F.col("order_count") > 5)
    .orderBy(F.desc("order_count"))
)

display(high_frequency.limit(10))

---
> ⊕ **BONUS — Joining DataFrames**
>
> Skip in standard schedule. Participants know JOIN from SQL. Cover it before the optional Workshop Task 12,
> which checks the row-duplication effect shown here.
---

## Joining DataFrames

> **Definition:** `df.join(other, condition, how)` combines two DataFrames; `how` is `"inner"` (default), `"left"`, `"right"` or `"full"`.
>
> **SQL:** `SELECT o.order_id, o.customer_id, c.first_name, c.country, o.total_amount FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id`

When both DataFrames have a column with the same name (here `customer_id`), refer to it through its DataFrame — `orders_df["customer_id"]` — both in the join condition and in `select()`, otherwise Spark reports an *ambiguous reference*.

In [ ]:
# Left join: keep every order, attach customer attributes where customer_id matches
orders_with_customers = (
    orders_df
    .join(customers_df, orders_df["customer_id"] == customers_df["customer_id"], "left")
    .select(
        orders_df["order_id"],
        orders_df["customer_id"],
        customers_df["first_name"],
        customers_df["country"],
        orders_df["total_amount"],
    )
)

display(orders_with_customers.limit(10))

In [ ]:
# Joined DataFrame works like any other — aggregate it
revenue_by_country = (
    orders_with_customers
    .groupBy("country")
    .agg(F.count("*").alias("order_count"), F.sum("total_amount").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)

display(revenue_by_country)

> **Why is `order_count` higher than the number of orders?**
>
> `orders_df` has **100,000** orders, but the join returns **more** rows. `customers_df` contains duplicate `customer_id` values
> (two different people share one ID), so every order of such a customer is matched **twice** — and its `total_amount` is counted twice.
>
> A join multiplies rows whenever the key on the right side is not unique. Check it with
> `customers_df.groupBy("customer_id").count().filter("count > 1")` — and deduplicate (`dropDuplicates(["customer_id"])`) before joining in Silver.
> Workshop Task 12 checks for exactly this effect.

---
> ⊕ **BONUS — Global Temporary Views**
>
> Skip in standard schedule. Global Temp Views share state across sessions within the same cluster.
> They are rarely needed at Fundamental level — standard Temp Views (`createOrReplaceTempView`) cover 99% of use cases.
---

### Global Temporary Views

> **Definition:** `df.createOrReplaceGlobalTempView("name")` registers a view visible to all sessions on the same classic cluster, under the `global_temp` schema (`SELECT * FROM global_temp.name`). It lives until the cluster restarts. Serverless compute does not support it.

In [0]:
# Create global temporary view
# Serverless compute (incl. Free Edition) does not support global temp views — use a temp view or a UC view
if IS_SERVERLESS:
    print("Serverless compute: global temporary views are not supported — this BONUS needs a classic cluster")
else:
    customers_df.createOrReplaceGlobalTempView("global_customers")

In [0]:
# Query global temp view
if not IS_SERVERLESS:
    global_result = spark.sql("""
        SELECT customer_segment, COUNT(*) as count
        FROM global_temp.global_customers
        GROUP BY customer_segment
        ORDER BY count DESC
        LIMIT 5
    """)

    display(global_result)

---
> ⊕ **BONUS — Complex SQL and SQL vs DataFrame API**
>
> Skip in standard schedule. A longer JOIN + GROUP BY + HAVING query and a short comparison of the two APIs.
---

### Complex SQL (JOINs + Aggregations)

Combining JOINs with GROUP BY and HAVING in a single SQL query.

In [0]:
# Example: Complex query with JOIN and aggregation
# customer_id is not unique in the raw customers file — two different people can share an ID,
# so such an ID appears twice below with the same order totals. Silver would deduplicate first.

complex_query = spark.sql("""
    SELECT 
        c.customer_id,
        c.first_name,
        c.last_name,
        c.country,
        COUNT(o.order_id) as total_orders,
        ROUND(SUM(o.total_amount), 2) as total_spent,
        ROUND(AVG(o.total_amount), 2) as avg_order_value,
        MAX(o.total_amount) as largest_order
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.first_name, c.last_name, c.country
    HAVING COUNT(o.order_id) > 0
    ORDER BY total_spent DESC
    LIMIT 10
""")

display(complex_query)

| Criteria | SQL | DataFrame API |
|---|---|---|
| **Best for** | JOINs, ad-hoc analysis, reporting | Pipelines with reusable logic |
| **Team** | SQL-first, analysts | Engineers, ML |
| **Performance** | Same engine and optimizer | Same engine and optimizer |